# 6. Mixture of Experts (MoE) Uzmanlar Karması

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# 1. Adım: Uzman (Expert) Modülünü Tanımlama
# Her bir uzman, basit bir Feed-Forward Network (FFN) olacak.
class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Basit bir iki katmanlı ağ: Linear -> ReLU -> Linear
        hidden = F.relu(self.fc1(x))
        output = self.fc2(hidden)
        return output

# 2. Adım: Yönlendirici (Gating Network) Modülünü Tanımlama
# Bu ağ, hangi uzmanın hangi token için ne kadar uygun olduğuna karar verir.
class GatingNetwork(nn.Module):
    def __init__(self, input_dim, num_experts):
        super().__init__()
        # Çıktısı, uzman sayısı kadar olan basit bir lineer katman
        self.layer = nn.Linear(input_dim, num_experts)

    def forward(self, x):
        # Girdiyi alır ve her uzman için bir "puan" (logit) üretir.
        logits = self.layer(x)
        return logits

# 3. Adım: MoE Katmanını Bir Araya Getirme
class MoELayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Uzmanları bir liste içinde oluştur
        # nn.ModuleList, PyTorch'un bu uzmanları modelin bir parçası olarak görmesini sağlar.
        self.experts = nn.ModuleList([
            Expert(input_dim, hidden_dim, output_dim) for _ in range(num_experts)
        ])
        
        # Yönlendirici ağı oluştur
        self.gating_network = GatingNetwork(input_dim, num_experts)
        print(f"{num_experts} uzmanlı ve top_k={top_k} olacak şekilde bir MoE katmanı oluşturuldu.")

    def forward(self, x):
        # x'in şekli: (batch_size, sequence_len, input_dim)
        # Eğitim kolaylığı için batch_size=1 ve sequence_len=token sayısı varsayalım.
        # x'in şekli: (num_tokens, input_dim)
        num_tokens, input_dim = x.shape

        # Adım A: Yönlendiriciden her token için uzman puanlarını al
        # gating_logits şekli: (num_tokens, num_experts)
        gating_logits = self.gating_network(x)
        
        # Adım B: Her token için en iyi k uzmanı ve onların puanlarını seç
        # torch.topk, hem değerleri (puanları) hem de indisleri döndürür.
        top_k_logits, top_k_indices = torch.topk(gating_logits, self.top_k, dim=-1)
        
        # Adım C: Seçilen k uzmanın puanlarına softmax uygulayarak ağırlıklar elde et
        # Bu ağırlıkların toplamı her token için 1 olur.
        top_k_weights = F.softmax(top_k_logits, dim=-1)

        # Adım D: Çıktıyı hesapla
        final_output = torch.zeros_like(x)

        # Her bir token için adım adım işlem yapalım (anlaşılırlık için döngü kullanıyoruz)
        print("\n--- Token Bazında Uzman Seçimi ve Hesaplama ---")
        for i in range(num_tokens):
            print(f"\nToken {i+1} için analiz:")
            print(f"  Tüm uzmanlar için ham puanlar (logits): {np.round(gating_logits[i].detach().numpy(), 2)}")
            print(f"  Seçilen en iyi {self.top_k} uzman (indisler): {top_k_indices[i].tolist()}")
            print(f"  Bu uzmanların ağırlıkları (softmax sonrası): {np.round(top_k_weights[i].detach().numpy(), 2)}")

            token_output = torch.zeros(input_dim)
            
            # Seçilen k uzmanı döngüye al
            for j in range(self.top_k):
                expert_index = top_k_indices[i, j].item()
                expert_weight = top_k_weights[i, j]
                
                # İlgili uzmandan çıktıyı al
                expert_output = self.experts[expert_index](x[i])
                
                # Ağırlıklı çıktıyı topla
                token_output += expert_weight * expert_output

                print(f"    -> Uzman {expert_index} çalıştı. Ağırlık: {expert_weight:.2f}")

            final_output[i] = token_output
        
        return final_output

if __name__ == '__main__':
    # Simülasyon için parametreler
    # Genellikle bir cümlenin embedding vektörlerinin boyutu
    input_dim = 32    
    # Her uzmanın içindeki gizli katman boyutu
    hidden_dim = 64   
    # Çıktı boyutu (girdi ile aynı)
    output_dim = 32   
    # Toplam uzman sayısı
    num_experts = 8   
    # Her token için seçilecek en iyi uzman sayısı
    top_k = 2         

    # Modeli oluştur
    moe_layer = MoELayer(input_dim, hidden_dim, output_dim, num_experts, top_k)

    # 4 farklı token'dan oluşan sahte bir girdi oluşturalım
    # (Örn: "Yapay zeka harikadır" cümlesinin token vektörleri)
    num_tokens = 4
    input_tokens = torch.randn(num_tokens, input_dim)

    # Modeli çalıştır ve çıktıyı al
    final_output = moe_layer(input_tokens)

    print("\n--- Nihai Sonuç ---")
    print(f"MoE katmanından geçen {num_tokens} token'ın nihai çıktı vektörleri:")
    # Çıktının sadece ilk 5 boyutunu gösterelim
    print(np.round(final_output[:, :5].detach().numpy(), 3))

8 uzmanlı ve top_k=2 olacak şekilde bir MoE katmanı oluşturuldu.

--- Token Bazında Uzman Seçimi ve Hesaplama ---

Token 1 için analiz:
  Tüm uzmanlar için ham puanlar (logits): [ 0.11 -0.1   0.22 -0.21 -1.05 -0.05  0.26 -0.47]
  Seçilen en iyi 2 uzman (indisler): [6, 2]
  Bu uzmanların ağırlıkları (softmax sonrası): [0.51 0.49]
    -> Uzman 6 çalıştı. Ağırlık: 0.51
    -> Uzman 2 çalıştı. Ağırlık: 0.49

Token 2 için analiz:
  Tüm uzmanlar için ham puanlar (logits): [-0.63  0.72 -0.67 -0.86  1.36 -0.2  -0.85  1.28]
  Seçilen en iyi 2 uzman (indisler): [4, 7]
  Bu uzmanların ağırlıkları (softmax sonrası): [0.52 0.48]
    -> Uzman 4 çalıştı. Ağırlık: 0.52
    -> Uzman 7 çalıştı. Ağırlık: 0.48

Token 3 için analiz:
  Tüm uzmanlar için ham puanlar (logits): [ 1.25 -0.05 -0.63  0.18  0.28 -0.53  0.17 -0.29]
  Seçilen en iyi 2 uzman (indisler): [0, 4]
  Bu uzmanların ağırlıkları (softmax sonrası): [0.73 0.27]
    -> Uzman 0 çalıştı. Ağırlık: 0.73
    -> Uzman 4 çalıştı. Ağırlık: 0.27

Token 